<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu defterin Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html)

# Week 03: Deep learning and CNN fundamentals

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lab.html) · [PDF notes](Week03_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/NB03_deep_learning_and_cnns.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. Training the networks of this notebook takes from a few seconds to about a minute per cell on an ordinary laptop, and no graphics card is needed. Run Section 0 first, whichever part you choose. It prepares the course files and the packages.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. PyTorch is the deep learning package of the course. Colab includes it; for Anaconda, SETUP.md shows the one-line installation. The second cell also downloads the cine cardiac magnetic resonance files of ACDC in the three groups of the week, about 105 MB, from the public database of the Human Heart Project (CREATIS, Lyon). It takes a minute or two and is skipped when the files are already there. The data are free for non-commercial scientific research under CC BY-NC-SA 4.0; every use cites Bernard et al., IEEE Transactions on Medical Imaging, 2018.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math

from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from vclinic import acdc, echonet
from vclinic.nifti import read_nifti

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError("PyTorch is not installed. Install it as SETUP.md describes (pip install torch), then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 4, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
GROUPS = ("NOR", "DCM", "HCM")
acdc.download(GROUPS)
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["ipywidgets", "torchvision", "cv2"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 3.1": (608256, 0),
    "Exercise 3.2": (104.5, 0.05),
    "Exercise 3.3": (0.323, 0.001),
    "Exercise A": (0.859, 0.001),
    "Exercise B": (-0.5, 0.001),
    "Exercise C": (90, 0),
    "Exercise D": (31, 0),
    "Exercise E": (12219, 0),
    "Exercise F": (1.429, 0.001),
    "Exercise G": (0.933, 0.001),
    "Exercise H": (4640, 0),
    "Exercise I": (73984, 0),
    "Exercise J": (0.933, 0.15),
    "Step 1.1": (19.8, 0.05),
    "Step 1.2": (62.2, 0.05),
    "Step 1.3": (0.967, 0.001),
    "Step 2.1": (0.2961, 0.0005),
    "Step 2.2": (12219, 0),
    "Step 2.3": (0.933, 0.15),
    "Step 2.4": (0.933, 0.15),
})
print("Expected answers are ready.")

In [ ]:
#@title Tables of the exercise prompts (run this cell; the Python path does not need to read it)
def exercise_table(key):
    """The table "data" of the prompt of an exercise, built from the tables of the Python path."""
    g = globals()
    k = key
    if k == "3.1":
        v = g["vol"]
        return pd.DataFrame({"rows": [v.shape[0]], "columns": [v.shape[1]], "slices": [v.shape[2]]})
    if k == "3.2":
        return pd.DataFrame(g["img"])
    if k == "3.3":
        return pd.DataFrame({"LVEDVI": g["Xn"][:, 0], "thickest wall": g["Xn"][:, 1], "label": g["tn"].astype(int)})
    if k in ("C", "E"):
        names = {"train": "training", "val": "validation", "test": "test"}
        t = pd.DataFrame({"patient": g["P"]["patient"], "set": [names[s] for s in g["P"]["set"]]})
        if k == "E":
            t.insert(1, "group", [g["P"]["classes"][c] for c in g["P"]["y"]])
            t["end-diastole"] = list(g["P"]["X"][:, 0])
            t["end-systole"] = list(g["P"]["X"][:, 1])
        return t
    if k == "J":
        net = g["course_model"]
        net.eval()
        with torch.no_grad():
            prob = F.softmax(net(g["Xte"]), dim=1).numpy()
        groups = ["NOR", "DCM", "HCM"]
        return pd.DataFrame({"patient": g["P"]["patient"][g["te"]], "true group": [groups[c] for c in g["P"]["y"][g["te"]]],
                             "NOR": prob[:, 0], "DCM": prob[:, 1], "HCM": prob[:, 2]})
    raise KeyError(k)
print("The tables of the exercise prompts are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**"data" and "answer".** As in Week 1, the data cell of each step prepares the table "data" that the prompt describes. The assistant keeps the result under the name "answer", and the check cell reads it from there.

**At the start of every stop** run its preparation cell once. It loads the data of the stop, so a stop can also be started on another day.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: Cine MR as numbers

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 3*

The expert contours of ACDC turn each examination into numbers: volumes, the ejection fraction (EF) and the thickness of the walls. In this stop the three groups are described with these numbers, the EF of patient 134 is computed from the voxels of the contours, and the rules of ACDC are applied to every patient.

**At the end of this stop:** The mean EF of the group DCM (19.8 percent), the EF of patient 134 (62.2 percent) and the accuracy of the rules of ACDC on the 90 patients (0.967).

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop)
GROUPS = ("NOR", "DCM", "HCM")
acdc_table = acdc.table(GROUPS)
case_dir = acdc.default_folder() / "patient134"
info = acdc.read_info(case_dir / "Info.cfg")
gt_ed, spacing = read_nifti(case_dir / f"patient134_frame{int(info['ED']):02d}_gt.nii.gz")
gt_es, _ = read_nifti(case_dir / f"patient134_frame{int(info['ES']):02d}_gt.nii.gz")
print(acdc_table.shape, gt_ed.shape, spacing)
print("Stop 1 is ready.")

### Step 1.1: The ejection fraction of the three groups

**Goal.** See how the EF separates the three groups of ACDC.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: In the cardiac MR data set ACDC, the cardiologist wants to see how the left ventricular ejection fraction differs between normal hearts, dilated cardiomyopathy and hypertrophic cardiomyopathy.  
> Data: The notebook holds a table named "data" with the 90 patients of ACDC in the groups NOR (normal), DCM (dilated cardiomyopathy) and HCM (hypertrophic cardiomyopathy). Each row is a patient: "patient" gives the name, "group" the group, "LVEF" the left ventricular ejection fraction in percent, "LVEDVI" the left ventricular end-diastolic volume indexed to the body surface area in mL/m², and "thickest wall" the thickest segment of the wall at end-diastole in mm. All were measured on the contours of the experts.  
> Task: Compute the mean LVEF of each group and print the three means with one decimal.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Keep the mean LVEF of the group DCM, with one decimal, under the name "answer".  
> Test: The mean of the group DCM must be below 40 percent, and that of the group HCM above 50 percent.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
data = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "patient", "group": "group", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "thickest wall"})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
data = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "patient", "group": "group", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "thickest wall"})
means = data.groupby("group")["LVEF"].mean().round(1)
print(means)
answer = float(means["DCM"])
print("Mean EF of DCM:", answer)
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** The group DCM has a mean EF of 19.8 percent, the normal group 62.4 and the group HCM 69.8 percent. A low EF is part of the definition of DCM in ACDC; in HCM the EF is normal or high, and the walls make the difference.

### Step 1.2: The ejection fraction of patient 134 from the voxels

**Goal.** Compute volumes and the EF from the expert contours slice by slice, as the software of a CMR laboratory does.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compute the end-diastolic and end-systolic volumes and the ejection fraction of the left ventricle of patient 134 of ACDC by summing the contoured cavity slice by slice.  
> Data: The notebook holds a table named "data". Each row is a short-axis slice of the examination, from the base to the apex: "slice" gives its number, and "end-diastole" and "end-systole" give the number of pixels inside the contour of the left ventricular cavity in that slice. One pixel measures 1.445 × 1.445 mm, and the slices are 10 mm thick.  
> Task: For each phase, add up the pixels of all slices and multiply the sum by the volume of one voxel, 1.445 × 1.445 × 10 mm³; divide by 1,000 to get mL. This gives the end-diastolic volume EDV and the end-systolic volume ESV. The EF is (EDV − ESV)/EDV × 100.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print EDV and ESV in mL and the EF in percent with one decimal, and keep the EF under the name "answer".  
> Test: EDV must be larger than ESV and lie between 50 and 300 mL.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"slice": np.arange(1, gt_ed.shape[2] + 1),
                     "end-diastole": [int((gt_ed[:, :, k] == 3).sum()) for k in range(gt_ed.shape[2])],
                     "end-systole": [int((gt_es[:, :, k] == 3).sum()) for k in range(gt_es.shape[2])]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"slice": np.arange(1, gt_ed.shape[2] + 1),
                     "end-diastole": [int((gt_ed[:, :, k] == 3).sum()) for k in range(gt_ed.shape[2])],
                     "end-systole": [int((gt_es[:, :, k] == 3).sum()) for k in range(gt_es.shape[2])]})
voxel_ml = 1.445 * 1.445 * 10 / 1000
edv = data["end-diastole"].sum() * voxel_ml
esv = data["end-systole"].sum() * voxel_ml
answer = round(100 * (edv - esv) / edv, 1)
print("EDV:", round(edv, 1), "mL · ESV:", round(esv, 1), "mL · EF:", answer, "%")
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** An EF of 62.2 percent, from 5,319 voxels at end-diastole and 2,010 at end-systole. The contours count every voxel of the cavity, slice by slice; Week 4 estimates the same volume on echocardiograms with the method of discs.

### Step 1.3: The rules of ACDC on every patient

**Goal.** Classify the 90 patients with the definitions of ACDC and see where the measurements alone go wrong.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how often the measurement rules of the ACDC definitions place a patient in the group given by the experts.  
> Data: The notebook holds a table named "data" with the 90 patients of ACDC in the groups NOR (normal), DCM (dilated cardiomyopathy) and HCM (hypertrophic cardiomyopathy). Each row is a patient: "patient" gives the name, "group" the group, "LVEF" the left ventricular ejection fraction in percent, "LVEDVI" the left ventricular end-diastolic volume indexed to the body surface area in mL/m², and "thickest wall" the thickest segment of the wall at end-diastole in mm. All were measured on the contours of the experts.  
> Task: Apply the rules of ACDC to every patient: The patient is DCM if LVEF is below 40 and LVEDVI above 100; otherwise HCM if the thickest wall is above 15 and LVEF is 50 or more; otherwise NOR. Print a table that counts the patients by group and by rule, and compute the share of patients whose rule equals their group.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the share with three decimals and keep it under the name "answer".  
> Test: The table must count 90 patients in total, and the share must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
data = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "patient", "group": "group", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "thickest wall"})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
data = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "patient", "group": "group", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "thickest wall"})


def acdc_rule(row):
    if row["LVEF"] < 40 and row["LVEDVI"] > 100:
        return "DCM"
    if row["thickest wall"] > 15 and row["LVEF"] >= 50:
        return "HCM"
    return "NOR"


data["rule"] = data.apply(acdc_rule, axis=1)
print(pd.crosstab(data["group"], data["rule"]))
answer = round(float((data["rule"] == data["group"]).mean()), 3)
print("Accuracy of the rules:", answer)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** The rules place 97 of 100 patients in their group. The three misses are borderline cases: one patient with DCM and an LVEDVI of 96.7 mL/m², one patient with DCM and an EF of 41.5 percent, and one patient with HCM whose thickest segment measures 14.6 mm on the expert contours. The experts weighed more than these three numbers.

**Stop 1 is complete.** Go back to the [lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#stop-1) and continue reading after the stop.

## Stop 2: A first convolutional network

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 4 to 7*

The network of the course learns the three groups from the pixels of the middle slices, with the end-diastolic and the end-systolic frame of each slice as two channels. In this stop the slices are normalised, the network is built, trained with early stopping and judged patient by patient on the 30 test patients.

**At the end of this stop:** A network with 12,219 parameters, its accuracy on the 30 test patients (about 0.933) and its macro F1 (about 0.933). Training takes about a minute.

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop)
GROUPS = ("NOR", "DCM", "HCM")
P = acdc.pairs(GROUPS)

class CmrCNN(nn.Module):
    """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            loss = F.cross_entropy(model(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history



def _course_decisions():
    """The decisions of the network of the course for the 30 test patients (trained once, about a minute)."""
    if "_course_result" in globals():
        return _course_result
    X, y, sets, patients = P["X"], P["y"], P["set"], P["patient"]
    tr, va, te = sets == "train", sets == "val", sets == "test"
    m, sd = float(X[tr].mean()), float(X[tr].std())
    T = lambda a: torch.tensor((a - m) / sd, dtype=torch.float32)
    torch.manual_seed(RANDOM_STATE)
    net = CmrCNN()
    train_model(net, T(X[tr]), torch.tensor(y[tr]), T(X[va]), torch.tensor(y[va]))
    net.eval()
    with torch.no_grad():
        prob = F.softmax(net(T(X[te])), dim=1).numpy()
    names = sorted(set(patients[te]))
    mean_prob = np.array([prob[patients[te] == n].mean(axis=0) for n in names])
    true = np.array([y[te][patients[te] == n][0] for n in names])
    globals()["_course_result"] = {"names": names, "true": true, "pred": mean_prob.argmax(axis=1), "slice_prob": prob}
    return _course_result
print("Stop 2 is ready.")

### Step 2.1: Normalising with the training slices

**Goal.** Prepare the inputs of the network with statistics that come from the training patients alone.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before a network learns the groups of ACDC from cine MR slices, the cardiologist wants to scale the pixels with numbers that come from the training patients alone.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients; "validation", 15 patients; "test", 30 patients). Each cell of the columns "end-diastole" and "end-systole" holds an image of 64 × 64 pixels with values from 0 to 1; one pixel is 2 mm.  
> Task: Compute the mean and the standard deviation of all pixels of both frames of the training slices, those whose set is "training". Normalise the pixels of all slices with these two numbers: Subtract the mean and divide by the standard deviation.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean and the standard deviation with four decimals, and keep the mean under the name "answer".  
> Test: After the normalisation, the training pixels must have a mean of about 0 and a standard deviation of about 1; the 135 training slices must keep their size of 64 × 64 pixels.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})
images = np.stack([np.stack(data["end-diastole"]), np.stack(data["end-systole"])], axis=1)
train = (data["set"] == "training").to_numpy()
mean, sd = float(images[train].mean()), float(images[train].std())
normalised = (images - mean) / sd
print(normalised[train].shape, "mean", round(mean, 4), "SD", round(sd, 4))
answer = round(mean, 4)
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** The training pixels have a mean of 0.2961. The validation and the test slices are normalised with the same two numbers, so that nothing of them flows into the training.

### Step 2.2: The network and its parameters

**Goal.** Build the network of the course and count its weights.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to build a small convolutional network for the groups of ACDC and to know how many weights it learns.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients; "validation", 15 patients; "test", 30 patients). Each cell of the columns "end-diastole" and "end-systole" holds an image of 64 × 64 pixels with values from 0 to 1; one pixel is 2 mm.  
> Task: Build a small convolutional network that receives the two frames of a slice as two channels. It has three blocks; each block has a convolution with 3 × 3 kernels that keeps the size of the image, batch normalisation, ReLU and a 2 × 2 max pooling. The blocks have 8, 16 and 32 kernels. After the blocks come flattening, dropout with a rate of 0.3 and one linear layer that gives a score to each of the three groups. Count the weights that the network learns, its trainable parameters.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of trainable parameters and keep it under the name "answer".  
> Test: For the first four training slices, the network must return four rows of three scores.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})


class CmrCNN(nn.Module):
    """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(RANDOM_STATE)
network = CmrCNN()
first4 = data[data["set"] == "training"].head(4)
x4 = torch.tensor(np.stack([np.stack(first4["end-diastole"]), np.stack(first4["end-systole"])], axis=1))
answer = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(tuple(network(x4).shape), "trainable parameters:", answer)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** 12,219 parameters, about half of them in the last linear layer, which turns the 2,048 values of the third block into three scores.

### Step 2.3: Training with early stopping and a decision per patient

**Goal.** Train the network and judge it as a clinic would: one decision for each patient.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to train a small convolutional network on the cine MR slices of ACDC and to judge it on the test patients as a clinic would, with one decision for each patient.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients; "validation", 15 patients; "test", 30 patients). Each cell of the columns "end-diastole" and "end-systole" holds an image of 64 × 64 pixels with values from 0 to 1; one pixel is 2 mm.  
> Task: Normalise the pixels with the mean and the standard deviation of the training slices. Build a small convolutional network that receives the two frames of a slice as two channels. It has three blocks; each block has a convolution with 3 × 3 kernels that keeps the size of the image, batch normalisation, ReLU and a 2 × 2 max pooling. The blocks have 8, 16 and 32 kernels. After the blocks come flattening, dropout with a rate of 0.3 and one linear layer that gives a score to each of the three groups. Train it on the training slices with the cross-entropy loss. After every epoch, measure the loss on the validation slices without learning from them, and keep the weights of the epoch with the lowest validation loss (early stopping). Make the random choices the same in every run by using the number 2026 as the starting value. Then compute the probabilities of the three groups for every test slice, average them over the three slices of each test patient, and take the group with the largest mean as the decision for the patient.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the epoch with the lowest validation loss and the share of the 30 test patients whose decision equals their group, with three decimals; keep the share under the name "answer".  
> Test: There must be 30 test patients, and the three mean probabilities of each patient must add up to 1.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})


class CmrCNN(nn.Module):
    """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            loss = F.cross_entropy(model(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history


images = np.stack([np.stack(data["end-diastole"]), np.stack(data["end-systole"])], axis=1)
codes = data["group"].map({"NOR": 0, "DCM": 1, "HCM": 2}).to_numpy()
sets = data["set"].to_numpy()
tr, va, te = sets == "training", sets == "validation", sets == "test"
mean, sd = float(images[tr].mean()), float(images[tr].std())
T = lambda a: torch.tensor((a - mean) / sd, dtype=torch.float32)
torch.manual_seed(RANDOM_STATE)
network = CmrCNN()
history = train_model(network, T(images[tr]), torch.tensor(codes[tr]), T(images[va]), torch.tensor(codes[va]))
network.eval()
with torch.no_grad():
    slice_prob = F.softmax(network(T(images[te])), dim=1).numpy()
test_patients = data["patient"].to_numpy()[te]
names = sorted(set(test_patients))
mean_prob = np.array([slice_prob[test_patients == n].mean(axis=0) for n in names])
true = np.array([codes[te][test_patients == n][0] for n in names])
answer = round(float((mean_prob.argmax(axis=1) == true).mean()), 3)
print("Best epoch:", int(np.argmin([h[1] for h in history])) + 1, "· test patients:", len(names), "· accuracy:", answer)
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** The course run classifies about 0.933 of the 30 test patients correctly. Training differs a little from one computer to the next, so the check accepts results within 0.15 of this value.

### Step 2.4: The confusion matrix and the macro F1

**Goal.** Read the result class by class and find the patients that the network misses. The data cell trains the network of the course once, which takes about a minute.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For the 30 test patients of ACDC, the cardiologist wants to read the decisions of a network group by group and to find the patients that it misses.  
> Data: The notebook holds a table named "data". Each row is a test patient: "patient" gives the name, "true group" the group given by the experts and "decision" the group chosen by the network; both are "NOR", "DCM" or "HCM".  
> Task: Build the confusion matrix with the true groups in the rows and the decisions in the columns, both in the order NOR, DCM, HCM, and print it. Compute the F1 score of each group and their plain mean, the macro F1. Print the names of the patients whose decision differs from their true group.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the macro F1 with three decimals and keep it under the name "answer".  
> Test: The confusion matrix must add up to 30 patients, and each of its rows to 10.  

In [ ]:
#@title Data of step 2.4: run this cell (it prepares the table "data" and shows it)
_r = _course_decisions()
data = pd.DataFrame({"patient": _r["names"], "true group": [P["classes"][k] for k in _r["true"]],
                     "decision": [P["classes"][k] for k in _r["pred"]]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.4
teach.check_step("Step 2.4", "answer", globals())

In [ ]:
#@title Ready code of step 2.4 (run it only if the code of the assistant does not pass the check)
_r = _course_decisions()
data = pd.DataFrame({"patient": _r["names"], "true group": [P["classes"][k] for k in _r["true"]],
                     "decision": [P["classes"][k] for k in _r["pred"]]})
order = ["NOR", "DCM", "HCM"]
cm = pd.crosstab(pd.Categorical(data["true group"], order), pd.Categorical(data["decision"], order),
                 rownames=["true group"], colnames=["decision"], dropna=False)
print(cm)
f1s = []
for g in order:
    tp = int(((data["true group"] == g) & (data["decision"] == g)).sum())
    fp = int(((data["true group"] != g) & (data["decision"] == g)).sum())
    fn = int(((data["true group"] == g) & (data["decision"] != g)).sum())
    f1s.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
print("F1:", {g: round(f, 3) for g, f in zip(order, f1s)})
print("Missed:", list(data.loc[data["true group"] != data["decision"], "patient"]))
answer = round(float(np.mean(f1s)), 3)
print("Macro F1:", answer)
teach.check_step("Step 2.4", "answer", globals())

**What the result shows.** A macro F1 of about 0.933. In the course run, the two errors are patients with HCM whom the network calls normal; one of them is patient 134, and the last section of the lecture page explains why.

**Stop 2 is complete.** Go back to the [lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#stop-2) and continue reading after the stop.

## Stop 3: Mini project: EchoNet-Dynamic, normal or abnormal (with your registration)

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 8 to 10*

The mini project of the session uses EchoNet-Dynamic, the echocardiography videos that each participant obtains through a personal registration at Stanford AIMI. Without a registration, follow the run of the instructor on the shared screen. With one, paste your personal download link into the first cell of this stop in the notebook, or put the zip file into the folder data/echonet of the course folder. The data stay in your Colab session or on your computer and never enter a prompt: The prompts describe only the table that the notebook prepares.

**At the end of this stop:** The number of videos of your copy, the share of test videos with an EF below 50 percent and the AUC of the network of the course for an abnormal EF. The checks compare your results with the ready code on the same copy.

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop)
# Paste your personal download link of EchoNet-Dynamic between the quotes, only if the data are not yet on this computer.
ECHONET_LINK = ""

class CmrCNN(nn.Module):
    """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

try:
    echo = echonet.find()
except FileNotFoundError:
    echo = echonet.download(ECHONET_LINK) if ECHONET_LINK else None
if echo is None:
    print("No copy of EchoNet-Dynamic was found. Register at Stanford AIMI, then paste your personal link above or put the zip file into the folder data/echonet of the course folder.")
else:
    print(echo)
print("Stop 3 is ready.")

### Step 3.1: The videos and their sets

**Goal.** Check that the copy is complete before any analysis.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before any analysis, the cardiologist wants to check that a copy of the echocardiography data set EchoNet-Dynamic, obtained through a personal registration, is complete.  
> Data: The notebook holds a table named "data" with the videos of the cardiologist's own copy of EchoNet-Dynamic, a data set of echocardiography videos. Each row is a video: "video" gives its name, "EF" the ejection fraction in percent, and "set" the set chosen by the authors ("training", "validation" or "test").  
> Task: Print the number of videos in each set and the total number of videos.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Keep the total number of videos under the name "answer".  
> Test: The set "training" must hold 7,465 videos, the set "validation" 1,288 and the set "test" 1,277.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    _ft = echonet.file_list(echo)
    data = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "set": _ft["Split"].map({"TRAIN": "training", "VAL": "validation", "TEST": "test"})})
    display(data)
globals().pop("answer", None)
pass

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    teach.check_value("Step 3.1", "answer", 10030, 0, globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    _ft = echonet.file_list(echo)
    data = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "set": _ft["Split"].map({"TRAIN": "training", "VAL": "validation", "TEST": "test"})})
    display(data)
    print(data["set"].value_counts())
    answer = int(len(data))
    print("Videos:", answer)
    teach.check_value("Step 3.1", "answer", 10030, 0, globals())

**What the result shows.** A complete copy holds 10,030 videos: 7,465 for training, 1,288 for validation and 1,277 for the test, the split of the authors.

### Step 3.2: The share of abnormal EF

**Goal.** Know how common the abnormal class is before judging a model.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before a model of EchoNet-Dynamic is judged, the cardiologist wants to know how common an abnormal ejection fraction, below 50 percent, is among the test videos.  
> Data: The notebook holds a table named "data" with the videos of the cardiologist's own copy of EchoNet-Dynamic, a data set of echocardiography videos. Each row is a video: "video" gives its name, "EF" the ejection fraction in percent, and "set" the set chosen by the authors ("training", "validation" or "test").  
> Task: Select the videos whose set is "test" and compute the share of them with an EF below 50 percent.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the share with three decimals and keep it under the name "answer".  
> Test: The share must lie between 0 and 1, and the number of test videos must be 1,277.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    _ft = echonet.file_list(echo)
    data = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "set": _ft["Split"].map({"TRAIN": "training", "VAL": "validation", "TEST": "test"})})
    display(data)
globals().pop("answer", None)
pass

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    teach.check_value("Step 3.2", "answer", round(float((echonet.file_list(echo).query('Split == "TEST"')["EF"] < 50).mean()), 3), 0.001, globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    _ft = echonet.file_list(echo)
    data = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "set": _ft["Split"].map({"TRAIN": "training", "VAL": "validation", "TEST": "test"})})
    display(data)
    test_table = data[data["set"] == "test"]
    answer = round(float((test_table["EF"] < 50).mean()), 3)
    print(len(test_table), "test videos; share with an EF below 50 percent:", answer)
    teach.check_value("Step 3.2", "answer", round(float((echonet.file_list(echo).query('Split == "TEST"')["EF"] < 50).mean()), 3), 0.001, globals())

**What the result shows.** The abnormal class is the smaller one. A model that calls every video normal would reach an accuracy of one minus this share, which is why the stop measures the AUC.

### Step 3.3: The network of the course on echocardiography

**Goal.** Train the same network on the end-diastolic and end-systolic frames of the videos and measure how well it recognises an abnormal EF. The data cell reads 1,400 videos, which takes a few minutes.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well a small convolutional network recognises an ejection fraction below 50 percent from two frames of an echocardiography video, the end-diastolic and the end-systolic one.  
> Data: The notebook holds a table named "data" with 1,000 training videos and 400 test videos of EchoNet-Dynamic. Each row is a video: "set" says "training" or "test", and "abnormal" is 1 when the EF is below 50 percent and 0 otherwise. Each cell of the columns "end-diastole" and "end-systole" holds the traced frame of that phase as an image of 64 × 64 pixels with values from 0 to 1.  
> Task: Normalise the pixels with the mean and the standard deviation of the training videos. Build a small convolutional network that receives the two frames of a video as two channels. It has three blocks; each block has a convolution with 3 × 3 kernels that keeps the size of the image, batch normalisation, ReLU and a 2 × 2 max pooling. The blocks have 8, 16 and 32 kernels. After the blocks come flattening, dropout with a rate of 0.3 and one linear layer that gives two scores, one for a normal and one for an abnormal EF. Train it on the training videos with the cross-entropy loss. Make the random choices the same in every run by using the number 2026 as the starting value. Then compute the probability of an abnormal EF for each test video and the area under the ROC curve (AUC).  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the AUC with three decimals and keep it under the name "answer".  
> Test: There must be 1,000 training videos with two frames of 64 × 64 pixels, and the AUC must lie between 0.5 and 1.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    _tr = echonet.frame_pairs(echo, split="TRAIN", n=1000, size=64, seed=RANDOM_STATE)
    _te = echonet.frame_pairs(echo, split="TEST", n=400, size=64, seed=RANDOM_STATE)
    data = pd.DataFrame({"set": ["training"] * len(_tr["abnormal"]) + ["test"] * len(_te["abnormal"]),
                         "abnormal": np.concatenate([_tr["abnormal"], _te["abnormal"]]).astype(int),
                         "end-diastole": list(np.concatenate([_tr["images"][:, 0], _te["images"][:, 0]])),
                         "end-systole": list(np.concatenate([_tr["images"][:, 1], _te["images"][:, 1]]))})
    display(data)
globals().pop("answer", None)
pass

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    teach.check_range("Step 3.3", "answer", 0.6, 1.0, globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
if echo is None:
    print("No copy of EchoNet-Dynamic yet: run the first cell of this stop with your personal link, or put the zip file into data/echonet.")
else:
    _tr = echonet.frame_pairs(echo, split="TRAIN", n=1000, size=64, seed=RANDOM_STATE)
    _te = echonet.frame_pairs(echo, split="TEST", n=400, size=64, seed=RANDOM_STATE)
    data = pd.DataFrame({"set": ["training"] * len(_tr["abnormal"]) + ["test"] * len(_te["abnormal"]),
                         "abnormal": np.concatenate([_tr["abnormal"], _te["abnormal"]]).astype(int),
                         "end-diastole": list(np.concatenate([_tr["images"][:, 0], _te["images"][:, 0]])),
                         "end-systole": list(np.concatenate([_tr["images"][:, 1], _te["images"][:, 1]]))})
    display(data)
    class CmrCNN(nn.Module):
        """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

        def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
            super().__init__()
            layers, c_in = [], in_channels
            for c_out in (8, 16, 32):
                layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
                layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
                layers += [nn.ReLU(), nn.MaxPool2d(2)]
                c_in = c_out
            self.features = nn.Sequential(*layers)
            self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

        def forward(self, x):
            return self.classifier(self.features(x))


    images = np.stack([np.stack(data["end-diastole"]), np.stack(data["end-systole"])], axis=1)
    train = (data["set"] == "training").to_numpy()
    labels = data["abnormal"].to_numpy()
    mean, sd = float(images[train].mean()), float(images[train].std())
    X_tr = torch.tensor((images[train] - mean) / sd, dtype=torch.float32)
    X_te = torch.tensor((images[~train] - mean) / sd, dtype=torch.float32)
    y_tr = torch.tensor(labels[train], dtype=torch.long)
    torch.manual_seed(2026)
    echo_net = CmrCNN(n_classes=2)
    optimiser = torch.optim.Adam(echo_net.parameters(), lr=0.003)
    for epoch in range(15):
        echo_net.train()
        order = torch.randperm(len(y_tr))
        for start in range(0, len(y_tr), 32):
            j = order[start:start + 32]
            loss = F.cross_entropy(echo_net(X_tr[j]), y_tr[j])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
    echo_net.eval()
    with torch.no_grad():
        p_abnormal = F.softmax(echo_net(X_te), dim=1)[:, 1].numpy()
    answer = round(float(roc_auc_score(labels[~train], p_abnormal)), 3)
    print(tuple(X_tr.shape), "AUC:", answer)
    teach.check_range("Step 3.3", "answer", 0.6, 1.0, globals())

**What the result shows.** Two frames of 64 × 64 pixels already carry much of the EF, because the change of the cavity between end-diastole and end-systole is visible. The result differs from one copy and one computer to the next, so the check accepts an AUC from 0.6 to 1.0. EchoNet-Dynamic itself reads the whole video at higher resolution and reached an AUC of 0.97 for a reduced EF.

**Stop 3 is complete.** Go back to the [lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#stop-3) and continue reading after the stop.

## Weekly project on the prompt path

*Project of the week: [README of Week 3, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project)
GROUPS = ("NOR", "DCM", "HCM")
P = acdc.pairs(GROUPS)
acdc_table = acdc.table(GROUPS)
_SETS = {"train": "training", "val": "validation", "test": "test"}

class CmrCNN(nn.Module):
    """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            loss = F.cross_entropy(model(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history



def _project_course():
    """The network of the course, trained once (about a minute), with the probabilities of the test slices."""
    if "_project_result" in globals():
        return _project_result
    X, y, sets, patients = P["X"], P["y"], P["set"], P["patient"]
    tr, va, te = sets == "train", sets == "val", sets == "test"
    m, sd = float(X[tr].mean()), float(X[tr].std())
    T = lambda a: torch.tensor((a - m) / sd, dtype=torch.float32)
    torch.manual_seed(RANDOM_STATE)
    net = CmrCNN()
    train_model(net, T(X[tr]), torch.tensor(y[tr]), T(X[va]), torch.tensor(y[va]))
    net.eval()
    with torch.no_grad():
        prob = F.softmax(net(T(X[te])), dim=1).numpy()
    names = sorted(set(patients[te]))
    mean_prob = np.array([prob[patients[te] == n].mean(axis=0) for n in names])
    globals()["_project_result"] = {"prob": prob, "names": names, "pred": mean_prob.argmax(axis=1),
                                    "true": np.array([y[te][patients[te] == n][0] for n in names])}
    return _project_result


def _project_middle_slices(name):
    """The three middle slices of a patient, the ones that the network saw."""
    d = acdc.default_folder() / name
    gt, _ = read_nifti(d / f"{name}_frame{int(acdc.read_info(d / 'Info.cfg')['ED']):02d}_gt.nii.gz")
    lv = [k for k in range(gt.shape[2]) if (gt[:, :, k] == 3).sum() > 0]
    mid = lv[len(lv) // 2]
    return sorted(sorted(lv, key=lambda k: (abs(k - mid), k))[:3])
print("The project is ready.")

### Project step 1: Train the network

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to train a small convolutional network on the cine MR slices of ACDC with early stopping, and to see how the losses of the training and the validation slices develop.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients; "validation", 15 patients; "test", 30 patients). Each cell of the columns "end-diastole" and "end-systole" holds an image of 64 × 64 pixels with values from 0 to 1; one pixel is 2 mm.  
> Task: Normalise the pixels with the mean and the standard deviation of the training slices. Build a small convolutional network that receives the two frames of a slice as two channels. It has three blocks; each block has a convolution with 3 × 3 kernels that keeps the size of the image, batch normalisation, ReLU and a 2 × 2 max pooling. The blocks have 8, 16 and 32 kernels. After the blocks come flattening, dropout with a rate of 0.3 and one linear layer that gives a score to each of the three groups. Train it on the training slices with the cross-entropy loss. Make the random choices the same in every run by using the number 2026 as the starting value. After every epoch, measure the loss on the validation slices without learning from them, and keep the weights of the epoch with the lowest validation loss. Draw the training and the validation loss against the epochs.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the figure of the two losses, and keep the epoch with the lowest validation loss under the name "answer".  
> Test: After the kept weights are loaded, the validation loss must again have its lowest value.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"patient": P["patient"], "group": [P["classes"][k] for k in P["y"]],
                     "set": [_SETS[s] for s in P["set"]],
                     "end-diastole": list(P["X"][:, 0]), "end-systole": list(P["X"][:, 1])})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: The confusion matrix of the network

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For the 30 test patients of ACDC, the cardiologist wants to see the confusion matrix of a network with one decision for each patient, and the precision, the recall and the F1 score of each group.  
> Data: The notebook holds a table named "data" with the 90 test slices, three for each of the 30 test patients. Each row is a slice: "patient" gives the patient, "true group" the group given by the experts, and "NOR", "DCM" and "HCM" the probabilities of the three groups, from 0 to 1, that the network of the course gives to the slice. The data cell trains this network once, which takes about a minute.  
> Task: Average the probabilities over the slices of each patient and take the group with the largest mean as the decision. Build the confusion matrix of the 30 patients with the true groups in the rows and the decisions in the columns, both in the order NOR, DCM, HCM. Compute the precision, the recall and the F1 score of each group, and the macro F1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the matrix and a table of the three measures by group, and keep the macro F1 under the name "answer".  
> Test: The matrix must add up to 30, and each of its rows to 10.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
_r = _project_course()
_te = P["set"] == "test"
data = pd.DataFrame({"patient": P["patient"][_te], "true group": [P["classes"][k] for k in P["y"][_te]],
                     "NOR": _r["prob"][:, 0], "DCM": _r["prob"][:, 1], "HCM": _r["prob"][:, 2]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: The rules on the same patients

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare the network with the measurement rules of ACDC on the same 30 test patients.  
> Data: The notebook holds a table named "data" with the 30 test patients of ACDC in the groups NOR (normal), DCM (dilated cardiomyopathy) and HCM (hypertrophic cardiomyopathy). Each row is a patient: "patient" gives the name, "group" the group, "LVEF" the left ventricular ejection fraction in percent, "LVEDVI" the left ventricular end-diastolic volume indexed to the body surface area in mL/m², and "thickest wall" the thickest segment of the wall at end-diastole in mm. All were measured on the contours of the experts.  
> Task: Apply the rules of ACDC: The patient is DCM if LVEF is below 40 and LVEDVI above 100; otherwise HCM if the thickest wall is above 15 and LVEF is 50 or more; otherwise NOR. Build the confusion matrix of group against rule, in the order NOR, DCM, HCM, and compute the accuracy.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the matrix and print the accuracy with three decimals, and keep the accuracy under the name "answer".  
> Test: The matrix must add up to 30.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
_t = acdc_table[acdc_table["split"] == "testing"]
data = _t[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "patient", "group": "group", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "thickest wall"})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: The errors

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For the test patients that a network of ACDC misclassifies, the cardiologist wants to see where the thickest wall of the left ventricle lies, from the base to the apex.  
> Data: The notebook holds a table named "data" with the misclassified test patients of the network of the course. Each row is one short-axis slice of one of these patients at end-diastole: "patient" gives the patient, "true group" and "decision" the group of the experts and the group of the network, "slice" the number of the slice from the base (0) to the apex, "thickest wall" the thickest segment of the wall in that slice in mm, and "seen by the network" says "yes" for the three middle slices, the only ones that the network saw.  
> Task: For every patient, plot the thickest wall against the number of the slice and mark the slices that the network saw. Print the slice with the thickest wall of the examination and whether the network saw it.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show one figure for each patient, and keep the number of misclassified patients under the name "answer".  
> Test: Each patient must have exactly three slices with "yes" in "seen by the network", and the number of patients must equal the number of errors in the confusion matrix of project step 2.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
_r = _project_course()
_rows = []
for _n, _t, _p in zip(_r["names"], _r["true"], _r["pred"]):
    if _t == _p:
        continue
    _seen = _project_middle_slices(_n)
    for _k, _w in acdc.walls(_n).items():
        _rows.append({"patient": _n, "true group": P["classes"][_t], "decision": P["classes"][_p],
                      "slice": _k, "thickest wall": _w,
                      "seen by the network": "yes" if _k in _seen else "no"})
data = pd.DataFrame(_rows)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 3: Images as arrays

For a computer, a grey-scale image is a table of numbers, one per pixel. A stack of images is an array with three dimensions, and a cine examination adds the time. This step builds an image by hand, reads a cine cardiac magnetic resonance volume of ACDC from its file, slices and normalises a stack of images and, at the end, writes the training loop of a single neuron. Every later section of the notebook uses these steps.

In [ ]:
# Python practice: a small example for learning the language
img7 = np.zeros((7, 7))
img7[1:6, 1:6] = 0.2      # a dark ring, like the myocardium
img7[2:5, 2:5] = 1.0      # a bright centre, like the blood of the cavity
print(img7.shape, img7.dtype)
print(img7)
plt.figure(figsize=(2.2, 2.2))
plt.imshow(img7, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()

**Python note.** `np.zeros((7, 7))` makes an array of 7 rows and 7 columns filled with 0. `img7[1:6, 1:6]` selects rows 1 to 5 and columns 1 to 5: In a slice `a:b`, the first number is included and the second is not. The first index is always the row, counted from the top, and the second the column. `plt.imshow` draws the array, and `vmin` and `vmax` fix which values are black and white. In cine cardiac magnetic resonance, the blood is bright and the myocardium dark.

In [ ]:
# Python practice: a small example for learning the language
case_dir = acdc.default_folder() / "patient134"
info = acdc.read_info(case_dir / "Info.cfg")
print(info)
vol, spacing = read_nifti(case_dir / f"patient134_frame{int(info['ED']):02d}.nii.gz")
print(vol.shape, vol.dtype, spacing)
mid = vol.shape[2] // 2
img = vol[:, :, mid].T.astype(float)
plt.figure(figsize=(3, 3.4))
plt.imshow(img, cmap="gray")
plt.title(f"patient134, ED, slice {mid}")
plt.axis("off")
plt.show()

**Python note.** `acdc.read_info` reads the small text file Info.cfg into a dictionary: the group, the height, the weight, the number of frames and the frames of end-diastole (ED) and end-systole (ES). `read_nifti` returns the voxels and their size in mm. The volume has three dimensions, (x, y, slice), and `vol[:, :, mid]` takes one slice. `.T` swaps rows and columns, so that the image appears with the rows of the scanner from top to bottom. `//` divides and drops the remainder.

In [ ]:
# Python practice: a small example for learning the language
S = acdc.slices(GROUPS)
print(S["images"].shape, S["classes"], S["pixel_mm"], "mm per pixel")
X12 = S["images"][:12]
print(X12.shape, S["patient"][:6], S["phase"][:6], S["group"][:6])
print(X12[0].shape, X12[:, 32, :].shape, round(float(X12[0, 32, 32]), 3))

**Python note.** `acdc.slices` cuts a square of 128 mm around the left ventricle from the three middle slices of every patient, at ED and at ES, and resamples it to 64 × 64 pixels. It returns a dictionary: `S["images"]` holds 540 images, and `S["patient"]`, `S["phase"]` and `S["group"]` say where each image comes from. `X12[0]` is the first image, and `X12[:, 32, :]` takes row 32 of every image. The colon alone means *all*.

In [ ]:
# Python practice: a small example for learning the language
x = X12[0]
z = (x - x.mean()) / x.std()                      # z-scores: mean 0, SD 1
m = (x - x.min()) / (x.max() - x.min())           # min-max: from 0 to 1
print(round(float(z.mean()), 3), round(float(z.std()), 3), round(float(m.min()), 3), round(float(m.max()), 3))
crop = x[16:48, 16:48]
print("crop", crop.shape)

**Python note.** Arithmetic on an array works on every element at once, so `x - x.mean()` subtracts the mean from all 4096 pixels. A network learns more easily from inputs near 0 with a spread near 1. For a network, the mean and the standard deviation (SD) must come from the training images only, as Section 3 shows. `x[16:48, 16:48]` crops the central 32 × 32 pixels.

In [ ]:
# Python practice: the training loop of one neuron
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


acdc_table = acdc.table(GROUPS)
feats = acdc_table[["lv_edvi", "max_wall_mm"]].to_numpy(float)
target = (acdc_table["group"] != "NOR").astype(int).to_numpy()     # 1: DCM or HCM
train_rows = (acdc_table["split"] == "training").to_numpy()
mu, sd = feats[train_rows].mean(axis=0), feats[train_rows].std(axis=0)
Xs = (feats - mu) / sd
Xn, tn = Xs[train_rows], target[train_rows]

w, b, lr = np.zeros(2), 0.0, 0.5
for step in range(201):
    p = sigmoid(Xn @ w + b)                                     # 1. forward pass
    loss = -np.mean(tn * np.log(p) + (1 - tn) * np.log(1 - p))  # 2. loss
    grad_w = Xn.T @ (p - tn) / len(tn)                          # 3. gradients
    grad_b = np.mean(p - tn)
    if step % 50 == 0:
        print(step, round(loss, 4), np.round(w, 3), round(b, 3))
    w = w - lr * grad_w                                         # 4. update
    b = b - lr * grad_b

**Python note.** Every training loop has the same four steps: the *forward pass* computes the predictions, the *loss* measures how wrong they are, the *gradients* say in which direction each weight should move to lower the loss, and the *update* moves the weights a small step, the learning rate, in that direction. The neuron learns from the 60 training patients whether a heart is normal or has a cardiomyopathy, from two standardised measurements. `Xn @ w` multiplies the 60 × 2 table by the two weights. `for step in range(201)` repeats the indented lines 201 times, and `step % 50 == 0` is true every 50 steps.

**Quick check.** `X12` has the shape (12, 64, 64). What is the shape of `X12[:, 10:20, 10:20]`?

<details><summary>Answer</summary>

(12, 10, 10): all 12 images, rows 10 to 19 and columns 10 to 19.

</details>

**Exercise 3.1.** How many voxels does `vol`, the ED volume of patient 134, hold in total?

In [ ]:
data = exercise_table("3.1")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(vol.size)
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** What is the mean value of the pixels of `img`, the middle slice of `vol`? Round to one decimal.

In [ ]:
data = exercise_table("3.2")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(img.mean()), 1)
teach.check("Exercise 3.2", answer)

**Exercise 3.3.** Train the neuron again from zero weights, but with the learning rate 0.05 instead of 0.5. What is the loss after 200 updates? Round to three decimals.

In [ ]:
data = exercise_table("3.3")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 3.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w3, b3 = np.zeros(2), 0.0
for step in range(200):
    p3 = sigmoid(Xn @ w3 + b3)
    w3 = w3 - 0.05 * Xn.T @ (p3 - tn) / len(tn)
    b3 = b3 - 0.05 * np.mean(p3 - tn)
p3 = sigmoid(Xn @ w3 + b3)
answer = round(float(-np.mean(tn * np.log(p3) + (1 - tn) * np.log(1 - p3))), 3)
teach.check("Exercise 3.3", answer)

## 1. A neuron and a network in NumPy

*Lecture: [Neuron, layer, activation function (a medical analogy)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#neural-networks-in-one-page)*

A neuron adds its inputs, each with its own weight, adds a bias and passes the sum through an *activation function*. A layer is a row of neurons that see the same inputs, and a network stacks layers so that each layer works on the outputs of the one before. Without a non-linear activation, a stack of layers would collapse into a single straight boundary.

**From the formulas to the code.** The weighted sum $z = \mathbf{w} \cdot \mathbf{x} + b$ is `w @ x + b`. The sigmoid $\sigma(z) = 1/(1 + e^{-z})$ is `sigmoid(z)`, the rectified linear unit (ReLU) $\max(0, z)$ is `np.maximum(0, z)`, and the softmax $e^{z_k}/\sum_j e^{z_j}$ turns the outputs of the last layer into probabilities that add up to 1. A layer with weights `W1` of shape (inputs, neurons) computes `x @ W1 + b1` for all its neurons at once.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(1)

In [ ]:
# Three activation functions
zz = np.linspace(-5, 5, 201)
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(zz, sigmoid(zz), color="#9B1B1B", lw=2, label="sigmoid")
ax.plot(zz, np.tanh(zz), color="#1F5F8B", lw=2, label="tanh")
ax.plot(zz, np.maximum(0, zz), color="#B97813", lw=2, label="ReLU")
ax.set_ylim(-1.2, 3)
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("z")
ax.legend()
plt.show()

In [ ]:
# A small network: 2 inputs, 3 hidden neurons with ReLU, 2 outputs with softmax
def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()


W1 = np.array([[1.5, -1.0, 0.5],
               [0.2, 1.2, -0.8]])
b1 = np.array([0.0, -0.5, 0.1])
W2 = np.array([[-1.0, 1.0],
               [-1.2, 1.2],
               [0.8, -0.8]])
b2 = np.array([0.6, -0.6])

x_case = Xs[(acdc_table["patient"] == "patient134").to_numpy()][0]   # standardised LVEDVI and thickest wall of the case
hidden = np.maximum(0, x_case @ W1 + b1)
out = softmax(hidden @ W2 + b2)
print("x =", np.round(x_case, 3))
print("hidden layer =", np.round(hidden, 3))
print("probabilities (normal, abnormal) =", np.round(out, 3))

**Python note.** `np.array([[...], [...]])` writes a table row by row: `W1` has 2 rows (inputs) and 3 columns (neurons). `x_case @ W1` gives one weighted sum per hidden neuron. Subtracting the largest value inside `softmax` changes nothing in the result, but it keeps `np.exp` from overflowing for large numbers. The weights here are set by hand, as in the worked example of the lecture page; Section 2 lets a network learn them.

**Observe and reflect.** Replace `np.maximum(0, ...)` by the identity, `x_case @ W1 + b1`, and work out on paper what the two layers become together. Why does a network need a non-linear activation between its layers?

**Exercise A.** A neuron receives the two standardised measurements `x = [1.2, -0.5]`. Its weights are `w = [0.8, -1.5]` and its bias `b = 0.1`, and it uses the sigmoid. What is its output? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
z_a = np.array([0.8, -1.5]) @ np.array([1.2, -0.5]) + 0.1
answer = round(float(sigmoid(z_a)), 3)
teach.check("Exercise A", answer)

## 2. Learning: Loss, gradients and the training loop in PyTorch

*Lecture: [Learning: Loss, gradients and the training loop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#learning-loss-gradients-and-the-training-loop)*

A network learns by lowering a *loss*, a number that grows when its predictions are wrong. *Backpropagation* applies the chain rule of calculus from the loss back to every weight and gives the *gradient*, the direction in which the loss rises fastest. *Gradient descent* moves each weight a small step against its gradient. PyTorch computes the gradients by itself: This is called *autograd*.

**From the formulas to the code.** For one patient with label $y$ and predicted probability $p$, the binary cross-entropy is $L = -[y \ln p + (1-y) \ln(1-p)]$, `F.binary_cross_entropy(p, y)`. For a neuron with the sigmoid, its gradient is $\partial L/\partial w_j = (p - y)\,x_j$. The update $w \leftarrow w - \eta\,\partial L/\partial w$ is what `optimizer.step()` does, after `loss.backward()` has filled `w.grad`. With several classes, `F.cross_entropy(logits, labels)` applies the softmax and the cross-entropy in one step.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(2)

In [ ]:
# Autograd checks the formula of the gradient
x_t = torch.tensor([1.2, -0.5])
w_t = torch.tensor([0.8, -1.5], requires_grad=True)
b_t = torch.tensor(0.1, requires_grad=True)
p_t = torch.sigmoid(w_t @ x_t + b_t)
loss_t = F.binary_cross_entropy(p_t, torch.tensor(1.0))
loss_t.backward()
print("p =", round(p_t.item(), 3), " loss =", round(loss_t.item(), 4))
print("autograd:", np.round(w_t.grad.numpy(), 4), round(b_t.grad.item(), 4))
print("formula (p − y) · x:", np.round((p_t.item() - 1.0) * x_t.numpy(), 4))

**Python note.** A tensor is PyTorch's array. `requires_grad=True` asks PyTorch to remember every operation done with the tensor, so that `loss_t.backward()` can go back through them and store the gradient in `w_t.grad`. `.item()` turns a tensor with one value into a Python number, and `.numpy()` turns a tensor into a NumPy array.

In [ ]:
# A network with 8 hidden neurons learns from the two measurements
Xn_t = torch.tensor(Xn, dtype=torch.float32)
tn_t = torch.tensor(tn)
torch.manual_seed(RANDOM_STATE)
mlp = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.05)
for step in range(301):
    loss = F.cross_entropy(mlp(Xn_t), tn_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(step, round(loss.item(), 4))

g1, g2 = np.meshgrid(np.linspace(-2.5, 5, 200), np.linspace(-2.5, 5, 200))
grid = np.c_[g1.ravel(), g2.ravel()]
with torch.no_grad():
    p_mlp = F.softmax(mlp(torch.tensor(grid, dtype=torch.float32)), dim=1)[:, 1].numpy().reshape(g1.shape)
p_one = sigmoid(grid @ w + b).reshape(g1.shape)
groups = acdc_table["group"].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, pr, name in [(axes[0], p_one, "one neuron"), (axes[1], p_mlp, "8 hidden neurons")]:
    ax.contourf(g1, g2, pr, levels=[0, 0.5, 1], colors=["#EAF1F7", "#F6E0E0"])
    for g, colour in [("NOR", "#1F5F8B"), ("DCM", "#9B1B1B"), ("HCM", "#B97813")]:
        ax.scatter(Xs[groups == g, 0], Xs[groups == g, 1], s=18, color=colour, label=g)
    ax.set_title(name)
    ax.set_xlabel("lv_edvi (z)")
axes[0].set_ylabel("max_wall_mm (z)")
axes[1].legend(loc="upper right")
plt.show()
test_rows = ~train_rows
with torch.no_grad():
    pred_mlp = mlp(torch.tensor(Xs[test_rows], dtype=torch.float32)).argmax(1).numpy()
print("test accuracy: one neuron", round(((sigmoid(Xs[test_rows] @ w + b) > 0.5) == target[test_rows]).mean(), 3),
      "| network", round((pred_mlp == target[test_rows]).mean(), 3))

**Python note.** `nn.Sequential` chains layers: `nn.Linear(2, 8)` is a layer of 8 neurons with 2 inputs each, then ReLU, then a layer of 2 output neurons. Each loop has the four steps of the Python step: the forward pass and the loss in one line, `optimizer.zero_grad()` to clear the old gradients, `loss.backward()` for the new ones and `optimizer.step()` for the update. `torch.no_grad()` turns off the bookkeeping of autograd when no learning is needed. Both models learn from the 60 training patients and are measured on the 30 test patients.

**Observe and reflect.** A heart is abnormal here when the ventricle is dilated *or* the wall is thick. Why can one neuron draw only one straight boundary in the left panel, while the network with a hidden layer can bend its boundary around the corner between the two groups of cardiomyopathy?

**The same model with the Keras Sequential API.** *Lecture: [A first model with the Keras Sequential API](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#a-first-model-with-the-keras-sequential-api)*

Keras describes the network of the previous cell as a list of layers, then compiles it with a loss, an optimiser and a measure, and fits it in one call. The cell runs Keras on PyTorch, so TensorFlow is not needed. Google Colab includes Keras; on a computer of one's own, `pip install keras` installs it.

In [ ]:
# The network of the previous cell with the Keras Sequential API
import os

os.environ.setdefault("KERAS_BACKEND", "torch")      # Keras runs on PyTorch
try:
    import keras

    keras.utils.set_random_seed(RANDOM_STATE)
    model_k = keras.Sequential([
        keras.Input(shape=(2,)),                     # two measurements
        keras.layers.Dense(8, activation="relu"),    # hidden layer of 8 neurons
        keras.layers.Dense(2, activation="softmax"), # normal or abnormal
    ])
    model_k.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model_k.summary()
    history_k = model_k.fit(Xn, tn, epochs=300, batch_size=len(Xn), verbose=0)
    loss_k, acc_k = model_k.evaluate(Xs[test_rows], target[test_rows], verbose=0)
    print("training loss after 300 epochs", round(history_k.history["loss"][-1], 4))
    print("test accuracy: Keras", round(acc_k, 3), "| PyTorch", round((pred_mlp == target[test_rows]).mean(), 3))
except ImportError:
    print("keras is not installed; pip install keras installs it. The PyTorch network above is the same model.")

**Python note.** `os.environ.setdefault("KERAS_BACKEND", "torch")` tells Keras to run on PyTorch, unless another backend was chosen before; it must come before `import keras`. `keras.Sequential([...])` takes the layers in order. `compile` fixes the loss, here `sparse_categorical_crossentropy` for labels written as 0 and 1, the optimiser and the reported measure. `fit` runs the whole training loop of the previous cell: `epochs=300` passes, each with one batch of all 60 training patients. `evaluate` returns the loss and the accuracy on the test patients, and `summary()` prints the layers with their 24 and 18 parameters.

**Exercise B.** One patient has `x = [1.0, 2.0]` and the label `y = 1`. A neuron with the sigmoid has the weights `w = [0.5, -0.25]` and the bias 0. Compute the binary cross-entropy with PyTorch, call `backward()` and give the gradient of the loss with respect to the first weight. Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w_b = torch.tensor([0.5, -0.25], requires_grad=True)
p_b = torch.sigmoid(w_b @ torch.tensor([1.0, 2.0]))
F.binary_cross_entropy(p_b, torch.tensor(1.0)).backward()
answer = round(float(w_b.grad[0]), 3)
teach.check("Exercise B", answer)

## 3. Cine MR of ACDC: Measurements and the input of the network

*Lecture: [Images as arrays: Cine cardiac MR of ACDC](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#images-as-arrays-cine-cardiac-mr-of-acdc)*

Each examination of ACDC comes with the expert contours of the left ventricular cavity, the myocardium and the right ventricular cavity at end-diastole (ED) and end-systole (ES). The contours give the volumes, the ejection fraction (EF) and the thickness of the walls, and the groups of ACDC are defined by them. The network of this week learns the groups from the pixels instead. Its input is prepared here: the ED and the ES frame of each slice as two channels, and a split of the patients into training, validation and test sets.

**From the formulas to the code.** A volume is the number of voxels inside a contour times the volume of one voxel, $V = N_{vox}\,\Delta x\,\Delta y\,\Delta z$: `(gt == 3).sum() * np.prod(spacing) / 1000` in mL. The EF is $(EDV - ESV)/EDV \times 100$. The normalisation $x' = (x - \mu_{train})/\sigma_{train}$ uses the mean and the SD of the training slices only. PyTorch expects images with the shape (number, channels, rows, columns), here (n, 2, 64, 64).

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(3)

In [ ]:
# The 90 patients of the three groups, measured on the expert contours
print(acdc_table.groupby("group")[["lv_ef", "lv_edvi", "max_wall_mm", "rv_ef"]].agg(["mean", "std"]).round(1))
print(acdc_table.groupby(["split", "group"]).size().unstack())

# The ejection fraction of patient 134 from the voxels of its contours
gt_ed, spacing = read_nifti(case_dir / f"patient134_frame{int(info['ED']):02d}_gt.nii.gz")
gt_es, _ = read_nifti(case_dir / f"patient134_frame{int(info['ES']):02d}_gt.nii.gz")
voxel_ml = float(np.prod(spacing)) / 1000
edv, esv = (gt_ed == 3).sum() * voxel_ml, (gt_es == 3).sum() * voxel_ml
print("patient134: EDV", round(edv, 1), "mL · ESV", round(esv, 1), "mL · EF", round(100 * (edv - esv) / edv, 1), "%")

In [ ]:
# A normal heart, DCM and HCM: the middle slice at ED with the contours
fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4))
for ax, (pid, g) in zip(axes, [("patient139", "NOR"), ("patient113", "DCM"), ("patient138", "HCM")]):
    rows = np.where((S["patient"] == pid) & (S["phase"] == "ED"))[0]
    i = rows[len(rows) // 2]
    ax.imshow(S["images"][i], cmap="gray", vmin=0, vmax=1)
    ax.contour(S["masks"][i] == 2, levels=[0.5], colors="#E8C020", linewidths=1)
    ax.contour(S["masks"][i] == 3, levels=[0.5], colors="#FF3030", linewidths=1.4)
    r = acdc_table.set_index("patient").loc[pid]
    ax.set_title(f"{pid} ({g})\nEF {r.lv_ef:.0f} % · LVEDVI {r.lv_edvi:.0f} · wall {r.max_wall_mm:.1f} mm", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Python note.** `(gt_ed == 3)` is an array of True and False, and `.sum()` counts the True values, here the voxels of the left ventricular cavity. `ax.contour(mask, levels=[0.5])` draws the border of a mask of 0 and 1. `acdc_table.set_index("patient").loc[pid]` finds the row of one patient, and `f"{r.lv_ef:.0f}"` writes a number without decimals. The images and the contours come from ACDC (CC BY-NC-SA 4.0).

In [ ]:
# The rules of ACDC on the measurements of the 90 patients
def acdc_rule(row):
    if row["lv_ef"] < 40 and row["lv_edvi"] > 100:
        return "DCM"
    if row["max_wall_mm"] > 15 and row["lv_ef"] >= 50:
        return "HCM"
    return "NOR"


acdc_table["rule"] = acdc_table.apply(acdc_rule, axis=1)
print(pd.crosstab(acdc_table["group"], acdc_table["rule"]))
print("accuracy of the rules:", round(float((acdc_table["rule"] == acdc_table["group"]).mean()), 3))
print(acdc_table.loc[acdc_table["rule"] != acdc_table["group"], ["patient", "group", "rule", "lv_ef", "lv_edvi", "max_wall_mm"]])

In [ ]:
# The input of the network: ED and ES as two channels, split by patient
P = acdc.pairs(GROUPS)
X, y, patients, sets = P["X"], P["y"], P["patient"], P["set"]
tr, va, te = sets == "train", sets == "val", sets == "test"
mean, sd = float(X[tr].mean()), float(X[tr].std())


def to_tensor(rows):
    return torch.tensor((X[rows] - mean) / sd, dtype=torch.float32)


Xtr, Xva, Xte = to_tensor(tr), to_tensor(va), to_tensor(te)
ytr, yva, yte = (torch.tensor(y[m], dtype=torch.long) for m in (tr, va, te))
for name, m in [("train", tr), ("val", va), ("test", te)]:
    print(name, "slices", int(m.sum()), "patients", len(set(patients[m])), np.bincount(y[m]))
print("mean and SD of the training pixels", round(mean, 4), round(sd, 4), tuple(Xtr.shape))

**Python note.** `acdc.pairs` returns the three middle slices of each patient with the ED frame in channel 0 and the ES frame in channel 1, and the set of each slice. The 30 testing patients of ACDC form the test set, and 5 patients of each group of the ACDC training set form the validation set. No patient appears in two sets. `np.bincount(y[m])` counts the slices of the classes 0, 1 and 2. The line with three names on the left unpacks three tensors made by a *generator expression*, a loop written inside parentheses.

**Exercise C.** How many slices does the test set hold?

In [ ]:
data = exercise_table("C")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(te.sum())
teach.check("Exercise C", answer)

## 4. Convolution by hand and with PyTorch

*Lecture: [The convolution operation: Finding edges and patterns in an image](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#convolution-filters-that-slide-over-an-image)*

A convolution slides a small table of weights, the *kernel*, over the image. At each position it multiplies the kernel with the pixels under it and adds the products, which gives one pixel of the output, the *feature map*. The same kernel is used everywhere, so a layer needs few weights, and a pattern is found wherever it appears. *Max pooling* keeps the largest value of each 2 × 2 block and halves the size.

**From the formulas to the code.** The output is $Y(i, j) = \sum_{u}\sum_{v} K(u, v)\,X(i+u, j+v) + b$, `F.conv2d(x, kernel, bias)`. With an input of $n$ pixels, a kernel of $k$, padding $p$ and stride $s$, the output has $\lfloor (n + 2p - k)/s \rfloor + 1$ pixels, `conv_out(n, k, s, p)`. Strictly, this formula is a cross-correlation; deep learning calls it convolution because the network learns the kernel either way.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(4)

In [ ]:
# A 3 × 3 kernel on a 5 × 5 patch, by hand and with PyTorch
patch = np.array([[0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 1, 0, 1, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 0, 0, 0]], dtype=float)
kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=float)          # vertical edges
by_hand = np.zeros((3, 3))
for r in range(3):
    for c in range(3):
        by_hand[r, c] = np.sum(patch[r:r + 3, c:c + 3] * kernel)
print(by_hand)
with_torch = F.conv2d(torch.tensor(patch)[None, None], torch.tensor(kernel)[None, None])
print(with_torch[0, 0].numpy())


def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


for n, k, s, p in [(64, 3, 1, 1), (64, 3, 1, 0), (64, 5, 2, 2), (32, 2, 2, 0)]:
    print(f"n={n} k={k} s={s} p={p} -> {conv_out(n, k, s, p)}")

**Python note.** Two nested `for` loops visit every position of the 3 × 3 output, and `patch[r:r + 3, c:c + 3]` is the 3 × 3 window at that position. `[None, None]` adds two dimensions in front, because `F.conv2d` expects (number, channels, rows, columns) for the image and (output channels, input channels, rows, columns) for the kernel.

In [ ]:
# Three kernels on the ED slice of patient 113, a patient with DCM
rows = np.where((S["patient"] == "patient113") & (S["phase"] == "ED"))[0]
slice_d = S["images"][rows[len(rows) // 2]]
kernels = {"vertical edges": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
           "horizontal edges": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
           "blur": [[1 / 9, 1 / 9, 1 / 9]] * 3}
t = torch.tensor(slice_d)[None, None]
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(slice_d, cmap="gray")
axes[0].set_title("patient113 (DCM)")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    fmap = F.conv2d(t, torch.tensor(k, dtype=torch.float32)[None, None], padding=1)[0, 0]
    ax.imshow(fmap.numpy(), cmap="RdBu_r")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.show()
print(tuple(t.shape), "→ max pooling →", tuple(F.max_pool2d(t, 2).shape))

**Python note.** A dictionary keeps names and values in pairs, and `.items()` returns the pairs one by one for the loop. `padding=1` adds a frame of zeros around the image, so a 3 × 3 kernel keeps the size at 64 × 64. The colour map `RdBu_r` shows negative values in blue and positive values in red, which is how an edge filter answers on the two sides of a wall: The image grows darker from the bright cavity into the dark myocardium.

**Pooling by hand.** *Lecture: [Pooling, dropout and batch normalisation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#pooling-dropout-batch-normalization)*

The cell pools the 4 × 4 feature map of the worked example of the lecture with max pooling and with average pooling, both over 2 × 2 blocks.

In [ ]:
# Max pooling and average pooling of a 4 × 4 feature map
fmap4 = torch.tensor([[1., 3., 0., 2.],
                      [4., 2., 1., 0.],
                      [0., 1., 5., 6.],
                      [2., 0., 3., 1.]])[None, None]
print("max pooling:\n", F.max_pool2d(fmap4, 2)[0, 0])
print("average pooling:\n", F.avg_pool2d(fmap4, 2)[0, 0])

**Python note.** `[None, None]` adds two dimensions in front, a batch of one image with one channel, because the pooling functions of PyTorch expect the shape (batch, channels, rows, columns). `F.max_pool2d(x, 2)` and `F.avg_pool2d(x, 2)` take blocks of 2 × 2 that do not overlap, and `[0, 0]` takes the one map back out.

**Exercise D.** A convolution with a 5 × 5 kernel, stride 2 and padding 1 runs over an input of 64 × 64 pixels. What is the size of one side of the output?

In [ ]:
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = conv_out(64, 5, 2, 1)
print(F.conv2d(torch.zeros(1, 1, 64, 64), torch.zeros(1, 1, 5, 5), stride=2, padding=1).shape)
teach.check("Exercise D", answer)

## 5. A first convolutional network

*Lecture: [A convolutional network for cine MR](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#a-convolutional-network-for-cine-mr)*

The network of the course has three blocks. Each block has a convolution with 3 × 3 kernels, batch normalisation, ReLU and max pooling. The image shrinks from 64 to 32, 16 and 8 pixels, while the number of feature maps grows from 2, the ED and the ES channel, to 8, 16 and 32. A linear layer turns the 32 × 8 × 8 = 2048 values into three outputs, one for each group. The training loop of Section 2 now works on *mini-batches* of 16 slices, and one pass over all training slices is an *epoch*. After each epoch, the loss on the validation slices decides which weights to keep.

**From the formulas to the code.** The class `CmrCNN` lists the layers in `__init__` and says in `forward` how an image passes through them. `train_model` runs the epochs and returns the training and validation loss of each epoch. `F.softmax(model(x), dim=1)` gives the probabilities of the three groups, and the decision for a patient is the group with the highest mean probability over the patient's slices.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(5)

In [ ]:
class CmrCNN(nn.Module):
    """Three convolution blocks and a linear layer; input: two frames of a slice as two channels."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


print(CmrCNN())
with torch.no_grad():
    print(CmrCNN()(Xtr[:4]).shape)

**Python note.** A *class* is a recipe for objects. `class CmrCNN(nn.Module)` builds on PyTorch's `nn.Module`, and `super().__init__()` lets the parent prepare itself first. `self.features` stores the layers inside the object. `layers += [...]` appends to a list, and `[...] if batchnorm else []` adds batch normalisation only when it is asked for. `nn.Sequential(*layers)` unpacks the list into separate arguments.

**Exercise E.** How many trainable parameters does `CmrCNN()` have?

In [ ]:
data = exercise_table("E")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in CmrCNN().parameters() if p.requires_grad)
teach.check("Exercise E", answer)

In [ ]:
def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE, augment_fn=None):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            xb = X_tr[j] if augment_fn is None else augment_fn(X_tr[j], order_gen)
            loss = F.cross_entropy(model(xb), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history


def by_patient(model, X_t, rows):
    """Mean probabilities of the slices of each patient, the true classes and the decisions."""
    model.eval()
    with torch.no_grad():
        prob = F.softmax(model(X_t), dim=1).numpy()
    names = sorted(set(patients[rows]))
    mean_p = np.array([prob[patients[rows] == n].mean(axis=0) for n in names])
    true = np.array([y[rows][patients[rows] == n][0] for n in names])
    return names, mean_p, true, mean_p.argmax(axis=1)

**Python note.** `model.train()` and `model.eval()` switch dropout and batch normalisation between their training and their evaluation behaviour. `torch.randperm` shuffles the order of the slices in every epoch. `copy.deepcopy(model.state_dict())` saves a full copy of the weights, and `load_state_dict` puts the best copy back at the end. The loss of each batch is weighted by its size, `len(j)`, so the last, smaller batch counts correctly. `by_patient` averages the probabilities of the three slices of each patient.

In [ ]:
# Training for 40 epochs with early stopping, about a minute
torch.manual_seed(RANDOM_STATE)
model = CmrCNN()
history = train_model(model, Xtr, ytr, Xva, yva, epochs=40)
best_epoch = int(np.argmin([h[1] for h in history])) + 1
names_te, prob_te, true_te, pred_te = by_patient(model, Xte, te)
print("best epoch", best_epoch, "· test patients", len(names_te),
      "· accuracy", round(float((pred_te == true_te).mean()), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(range(1, len(history) + 1), [h[0] for h in history], color="#1F5F8B", label="training")
ax.plot(range(1, len(history) + 1), [h[1] for h in history], color="#9B1B1B", label="validation")
ax.axvline(best_epoch, color="#888888", ls="--", lw=1)
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy loss")
ax.legend()
plt.show()

**Observe and reflect.** The course trained the same network with the same seed and reached 28 of 30 test patients, an accuracy of 0.933, with the best epoch 37. Do you get the same numbers? Small differences between computers come from the order of floating-point operations. Why does one patient more or less change the accuracy by 0.033 here, and what does this say about the precision of results on 30 patients?

## 6. Overfitting: Dropout, batch normalisation and augmentation

*Lecture: [Detecting and preventing overfitting](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#overfitting-dropout-batch-normalisation-and-augmentation)*

A network with twelve thousand weights can learn 135 training slices by heart. The training loss then keeps falling while the validation loss stops falling or rises. Dropout, batch normalisation, augmentation and early stopping work against it.

**From the formulas to the code.** Dropout keeps a unit with the probability $1 - p$ and divides it by $1 - p$ in training, $\tilde{a} = m\,a/(1 - p)$, `nn.Dropout(p)`. Batch normalisation computes $\hat{x} = (x - \mu_B)/\sqrt{\sigma_B^2 + \varepsilon}$ over the mini-batch and then $\gamma\hat{x} + \beta$, `nn.BatchNorm2d(c)`. The augmentation of the course turns a slice by 0, 90, 180 or 270 degrees, `torch.rot90`: A turn keeps the anatomy and changes only the direction of the view. A mirror image would reverse the sides of the heart, as in the rare mirror-image dextrocardia.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(6)

In [ ]:
# Augmentation: a slice of patient 134 turned in steps of a quarter circle
def augment(xb, gen):
    k = int(torch.randint(0, 4, (1,), generator=gen))
    return torch.rot90(xb, k, dims=(2, 3))


rows = np.where((S["patient"] == "patient134") & (S["phase"] == "ED"))[0]
case_slice = torch.tensor(S["images"][rows[1]])
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for k, ax in enumerate(axes):
    ax.imshow(torch.rot90(case_slice, k).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{90 * k}°")
    ax.axis("off")
plt.show()

**Python note.** `torch.rot90(x, k, dims=(2, 3))` turns the images of a batch by k quarter circles in the plane of their rows and columns. `torch.randint(0, 4, (1,), generator=gen)` draws one whole number from 0 to 3 with its own random generator, so that the same seed gives the same augmentation again. Each mini-batch is turned once, and over many epochs the network sees every slice in all four orientations.

**Image augmentation for cardiac images.** *Lecture: [Image augmentation for cardiac images](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#image-augmentation-for-cardiac-images)*

Turns are one family of augmentation. The cell shows two more, a lower contrast and a small shift, and the mirror image, which the course leaves out because it would reverse the sides of the heart. The training below uses the turns only.

In [ ]:
# Two more augmentations and the mirror image that the course leaves out
lower_contrast = 0.5 + 0.6 * (case_slice - 0.5)                  # contrast lowered by 40 percent
shifted = torch.roll(case_slice, shifts=(0, 6), dims=(0, 1))     # six pixels to the right
mirror = torch.flip(case_slice, dims=(1,))                        # not used: the sides of the heart change places
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, (img, title) in zip(axes, [(case_slice, "original"), (lower_contrast, "lower contrast"),
                                   (shifted, "shift of 6 pixels"), (mirror, "mirror, not used")]):
    ax.imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()

**Python note.** `0.5 + 0.6 * (x - 0.5)` pulls every pixel towards the middle grey and lowers the contrast by 40 percent. `torch.roll` shifts the image by six columns; the pixels that leave on the right come back on the left, which is acceptable for a small shift around a centred heart. `torch.flip(x, dims=(1,))` reverses the columns and gives the mirror image.

In [ ]:
# No regularisation against dropout with batch normalisation, 60 epochs each, about a minute
curves = {}
for name, kw in [("no regularisation", dict(dropout=0.0, batchnorm=False)),
                 ("dropout 0.5 and batch normalisation", dict(dropout=0.5, batchnorm=True))]:
    torch.manual_seed(RANDOM_STATE)
    net = CmrCNN(**kw)
    curves[name] = train_model(net, Xtr, ytr, Xva, yva, epochs=60)
    _, _, true_n, pred_n = by_patient(net, Xte, te)
    best = int(np.argmin([h[1] for h in curves[name]])) + 1
    print(name, "· best epoch", best, "· test accuracy", round(float((pred_n == true_n).mean()), 3))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), sharey=True)
for ax, (name, h) in zip(axes, curves.items()):
    ax.plot(range(1, 61), [v[0] for v in h], color="#1F5F8B", label="training")
    ax.plot(range(1, 61), [v[1] for v in h], color="#9B1B1B", label="validation")
    ax.axvline(int(np.argmin([v[1] for v in h])) + 1, color="#888888", ls="--", lw=1)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()
plt.show()

**Observe and reflect.** Without regularisation, the training loss falls almost to 0 while the validation loss rises after a few epochs: The network learns its 135 slices by heart. Which epoch would early stopping keep in each panel? Change `epochs=60` to 120 and run the cell again: What happens to the gap between the two curves, and to the test accuracy?

In [ ]:
# Dropout and batch normalisation on small tensors
drop = nn.Dropout(p=0.5)
drop.train()
torch.manual_seed(1)
print("dropout in training mode:", drop(torch.ones(8)).numpy())
drop.eval()
print("dropout in evaluation mode:", drop(torch.ones(8)).numpy())
bn = nn.BatchNorm1d(1)
values = torch.tensor([[9.0], [10.0], [12.0], [13.0]])
print("batch normalisation:", bn(values).detach().numpy().ravel().round(3))

**Exercise F.** In training mode, dropout with p = 0.3 sets 30 percent of the values to 0 on average. What value do the kept elements of `torch.ones(10)` take? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d03 = nn.Dropout(p=0.3)
d03.train()
out03 = d03(torch.ones(10))
answer = round(float(out03[out03 > 0][0]), 3)
teach.check("Exercise F", answer)

## 7. Reading the result: Precision, recall and F1

*Lecture: [Interpreting the confusion matrix and the F1 score](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#reading-the-result-precision-recall-and-f1)*

The accuracy of a network says how many patients it classifies correctly, but not which ones it misses. For each class, the *precision* is the share of correct decisions among the patients that the network places in the class, and the *recall* is the share of the patients of the class that it finds. The *F1 score* combines both, and the *macro F1* is the plain mean of the F1 of the classes.

**From the formulas to the code.** In a confusion matrix with the true classes in the rows and the decisions in the columns, $TP$ of class $k$ is the diagonal element `cm[k, k]`, $FP$ the rest of its column and $FN$ the rest of its row. $P = TP/(TP + FP)$, $R = TP/(TP + FN)$ and $F1 = 2\,TP/(2\,TP + FP + FN)$; `f1_score(true, pred, average="macro")` gives the macro F1.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(7)

In [ ]:
# The network of Section 5 on the 30 test patients, by hand with NumPy
cm = confusion_matrix(true_te, pred_te, labels=[0, 1, 2])
print(pd.DataFrame(cm, index=["NOR", "DCM", "HCM"], columns=["NOR", "DCM", "HCM"]))
tp = np.diag(cm)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp
precision = tp / np.maximum(tp + fp, 1)
recall = tp / np.maximum(tp + fn, 1)
f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
print(pd.DataFrame({"precision": precision, "recall": recall, "F1": f1}, index=["NOR", "DCM", "HCM"]).round(3))
print("macro F1", round(float(f1.mean()), 3), "· sklearn", round(f1_score(true_te, pred_te, average="macro"), 3))

**Python note.** `np.diag(cm)` takes the diagonal of the matrix, the correct decisions of each class. `cm.sum(axis=0)` adds each column and `cm.sum(axis=1)` each row. `np.maximum(..., 1)` keeps a division by zero away when a class receives no decision at all. The hand calculation and `f1_score` must agree.

In [ ]:
# The misclassified patients with their measurements
classes = ["NOR", "DCM", "HCM"]
errors = [(n, classes[t_], classes[p_], np.round(pr, 3)) for n, t_, p_, pr in zip(names_te, true_te, pred_te, prob_te) if t_ != p_]
measures = acdc_table.set_index("patient")
for n, t_, p_, pr in errors:
    r = measures.loc[n]
    print(n, "true", t_, "· network", p_, pr, "· EF", r.lv_ef, "· thickest wall", r.max_wall_mm, "mm")

**Observe and reflect.** The course run misses two patients with HCM and calls them normal; patient 134 is one of them. Both have a thickest segment above 15 mm, so the rules of ACDC on the measurements place them correctly. The network sees only the three middle slices. Where in the heart could the thickest segment lie, and how would you check it?

In [ ]:
# Patient 134: the thickest wall in every slice at ED
walls = acdc.wall_by_slice(gt_ed, spacing)
lv_slices = [k for k in range(gt_ed.shape[2]) if (gt_ed[:, :, k] == 3).any()]
mid_k = lv_slices[len(lv_slices) // 2]
seen = sorted(sorted(lv_slices, key=lambda k: (abs(k - mid_k), k))[:3])
print("slice, thickest wall (mm), from the base to the apex")
for k in lv_slices:
    print(k, round(walls[k], 1), "seen by the network" if k in seen else "")

**Python note.** `acdc.wall_by_slice` returns a dictionary: the number of each slice and the thickest point of its wall in mm. On every slice, it measures the distance from each point of the endocardial border to the nearest point outside the wall, on the epicardial side or in the right ventricle, and keeps the largest. Inside, `ndimage.distance_transform_edt` gives every pixel its distance to the nearest pixel outside the cavity and the wall. The slices of ACDC are numbered from the base, slice 0, to the apex.

**Exercise G.** A network gives the confusion matrix `cm3 = [[10, 0, 0], [0, 10, 0], [2, 0, 8]]` for NOR, DCM and HCM (true classes in the rows). What is its macro F1? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
cm3 = np.array([[10, 0, 0], [0, 10, 0], [2, 0, 8]])
tp3 = np.diag(cm3)
answer = round(float(np.mean(2 * tp3 / (2 * tp3 + (cm3.sum(axis=0) - tp3) + (cm3.sum(axis=1) - tp3)))), 3)
teach.check("Exercise G", answer)

## 8. Known architectures and skip connections

*Lecture: [The VGG16 and ResNet50 architectures (conceptual)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#known-architectures-from-lenet-to-resnet)*

Convolutional networks grew from LeNet-5, which read handwritten digits with about sixty thousand weights, to networks with millions of weights and more than a hundred layers. Very deep networks became trainable with the *skip connection* of ResNet: A block adds its input to its output, $y = F(x) + x$. The gradient can then flow past the block, and a block that has nothing to add can learn $F(x) = 0$.

**From the formulas to the code.** A convolution layer with $c_{in}$ input channels, $c_{out}$ output channels and a $k \times k$ kernel has $c_{in} \cdot c_{out} \cdot k^2 + c_{out}$ parameters, the last term for the biases. A batch normalisation layer has $2c$ parameters, $\gamma$ and $\beta$ for each of its $c$ channels. `count(model)` adds the sizes of all trainable tensors.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(8)

In [ ]:
# Parameters of LeNet-5, of a VGG block and of a residual block
def count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


lenet5 = nn.Sequential(nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2),
                       nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2), nn.Flatten(),
                       nn.Linear(16 * 5 * 5, 120), nn.Tanh(), nn.Linear(120, 84), nn.Tanh(), nn.Linear(84, 10))
vgg_block = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
                          nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))


class BasicBlock(nn.Module):
    """The basic block of ResNet: two 3 × 3 convolutions and a skip connection."""

    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)            # the skip connection


print("LeNet-5 (32 × 32):", count(lenet5), "|", tuple(lenet5(torch.zeros(1, 1, 32, 32)).shape))
print("VGG block 64 → 128:", count(vgg_block))
print("CmrCNN:", count(CmrCNN()))
try:
    import torchvision
    print("ResNet-18 (torchvision):", count(torchvision.models.resnet18(weights=None)))
    print("ResNet-50 (torchvision):", count(torchvision.models.resnet50(weights=None)))
except ImportError:
    print("torchvision is not installed; ResNet-18 has 11,689,512 and ResNet-50 25,557,032 parameters.")

**Python note.** `bias=False` leaves out the biases of the convolutions, because the batch normalisation right after them has its own shift $\beta$. In `forward`, `out + x` is the skip connection; it needs `out` and `x` to have the same shape, which the padding of 1 keeps. `weights=None` builds the networks of torchvision with random weights, so nothing is downloaded; Week 4 loads the weights learned on ImageNet.

In [ ]:
# VGG-16 and the bottleneck block of ResNet-50
plain = nn.Sequential(nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.Conv2d(256, 256, 3, padding=1, bias=False))
bottleneck = nn.Sequential(nn.Conv2d(256, 64, 1, bias=False),
                           nn.Conv2d(64, 64, 3, padding=1, bias=False),
                           nn.Conv2d(64, 256, 1, bias=False))
print("two 3 × 3 convolutions on 256 channels:", count(plain))
print("bottleneck 1 × 1, 3 × 3, 1 × 1:", count(bottleneck),
      "| share", round(count(bottleneck) / count(plain), 3))
try:
    import torchvision
    vgg16 = torchvision.models.vgg16(weights=None)
    print("VGG-16 (torchvision):", count(vgg16), "| in the three dense layers:", count(vgg16.classifier),
          "| share", round(count(vgg16.classifier) / count(vgg16), 3))
except ImportError:
    print("torchvision is not installed; VGG-16 has 138,357,544 parameters, 123,642,856 of them in its dense layers.")

**Python note.** A 1 × 1 convolution, `nn.Conv2d(256, 64, 1)`, mixes the 256 channels at each pixel into 64 new ones without looking at the neighbours, so it changes the number of channels cheaply. `vgg16.classifier` is the part of VGG-16 after the convolutions, its three dense layers, and `count` adds their parameters.

**Exercise H.** How many parameters does a convolution layer with 16 input channels, 32 output channels, a 3 × 3 kernel and biases have?

In [ ]:
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = count(nn.Conv2d(16, 32, 3))
teach.check("Exercise H", answer)

**Exercise I.** How many trainable parameters does `BasicBlock(64)` have?

In [ ]:
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = count(BasicBlock(64))
print(BasicBlock(64)(torch.randn(1, 64, 16, 16)).shape)
teach.check("Exercise I", answer)

## 9. Working with an AI assistant, part 3: Reviewing a training loop

*Lecture: [Working with an AI assistant, part 3: Reviewing a training loop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#ai-pair-programming-part-3-reviewing-a-training-loop)*

A participant asked an AI assistant to *evaluate the network of the course on the test slices*. The code below is what came back. It runs and reports an accuracy. Read it with the six questions of the training loop review in the lecture before you trust the number: Is the split made by patient? Where do the statistics of the normalisation come from? Is the model in evaluation mode, and are the gradients off? Does the output layer fit the loss? Does the measure suit the classes?

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(9)

In [ ]:
# Code from an AI assistant: evaluate the network of the course
course_model = CmrCNN()
course_model.load_state_dict(model.state_dict())
X_all = torch.tensor((X - X.mean()) / X.std(), dtype=torch.float32)
probs = F.softmax(course_model(X_all[te]), dim=1)
accuracy = (probs.argmax(dim=1).numpy() == y[te]).mean()
print("Accuracy:", round(float(accuracy), 3))

**Observe and reflect.** The code has three problems and one omission. The statistics of the normalisation come from all 270 slices, the test slices included. A new model starts in training mode, so dropout switches units off at random and batch normalisation uses the statistics of the test slices. Autograd also keeps track of every operation. The accuracy is counted on slices, while the clinic needs one decision per patient. Run the cell twice: Why does the accuracy change?

**Exercise J.** Evaluate `course_model` correctly: Switch it to evaluation mode, compute the probabilities of the test slices `Xte`, which were normalised with the training slices, inside `torch.no_grad()`, average them over the slices of each test patient and compute the share of the 30 test patients whose decision equals their class. Round to three decimals.

In [ ]:
data = exercise_table("J")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
_, _, true_j, pred_j = by_patient(course_model, Xte, te)
answer = round(float((pred_j == true_j).mean()), 3)
teach.check("Exercise J", answer)

The [prompt workshop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lab.html#part-f) of the interactive lab writes the prompt of such an evaluation with the seven fields of the prompt card. The field *Test* holds the check that catches the problems of the review: The same evaluation run twice must give the same numbers.

## 10. EchoNet-Dynamic with your own registration

*Lecture: [Mini project: Classifying normal and abnormal echocardiograms of the EchoNet-Dynamic data set](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#echonet-dynamic-echocardiography-with-personal-registration)*

EchoNet-Dynamic holds 10,030 apical four-chamber echocardiography videos of Stanford Medicine with the ejection fraction and the expert tracings at end-diastole and end-systole. Each participant registers at the Stanford AIMI shared data sets, accepts the Research Use Agreement and downloads the data personally. The cells below read that copy: Paste your personal download link into the first cell, or put the zip file into the folder `data/echonet` of the course folder. Without a copy, the cells only print a message.

**From the formulas to the code.** `echonet.file_list(echo)` reads FileList.csv, and `echonet.frame_pairs(echo, split, n, size)` reads the two traced frames of n videos as two channels of 64 × 64 pixels, with the label 1 for an EF below 50 percent. The network of Section 5 is used with two outputs, and the area under the ROC curve measures how well it ranks abnormal above normal videos.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(10)

In [ ]:
# Your copy of EchoNet-Dynamic
ECHONET_LINK = ""   # paste your personal download link here, only if the data are not yet on this computer
try:
    echo = echonet.find()
except FileNotFoundError:
    echo = echonet.download(ECHONET_LINK) if ECHONET_LINK else None
print(echo if echo is not None else "No copy of EchoNet-Dynamic was found; the next cells will wait for one.")

In [ ]:
# The videos, their sets and their ejection fraction
if echo is None:
    print("Register for EchoNet-Dynamic first (see the lecture page).")
else:
    file_table = echonet.file_list(echo)
    print(file_table["Split"].value_counts())
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    for split, colour in [("TRAIN", "#1F5F8B"), ("TEST", "#9B1B1B")]:
        ax.hist(file_table.loc[file_table["Split"] == split, "EF"], bins=40, alpha=0.6, density=True, color=colour, label=split)
    ax.axvline(50, color="#444444", ls="--", lw=1)
    ax.set_xlabel("EF (%)")
    ax.legend()
    plt.show()
    print("share with an EF below 50 percent in the test set",
          round(float((file_table.loc[file_table["Split"] == "TEST", "EF"] < 50).mean()), 3))

In [ ]:
# The network of the course on 1000 training and 400 test videos, a few minutes
if echo is None:
    print("Register for EchoNet-Dynamic first (see the lecture page).")
else:
    train_pairs = echonet.frame_pairs(echo, split="TRAIN", n=1000, size=64, seed=RANDOM_STATE)
    test_pairs = echonet.frame_pairs(echo, split="TEST", n=400, size=64, seed=RANDOM_STATE)
    e_mean, e_sd = float(train_pairs["images"].mean()), float(train_pairs["images"].std())
    Xe_tr = torch.tensor((train_pairs["images"] - e_mean) / e_sd, dtype=torch.float32)
    Xe_te = torch.tensor((test_pairs["images"] - e_mean) / e_sd, dtype=torch.float32)
    ye_tr = torch.tensor(train_pairs["abnormal"], dtype=torch.long)
    torch.manual_seed(RANDOM_STATE)
    echo_model = CmrCNN(n_classes=2)
    optimizer = torch.optim.Adam(echo_model.parameters(), lr=0.003)
    for epoch in range(15):
        echo_model.train()
        order = torch.randperm(len(ye_tr))
        for start in range(0, len(ye_tr), 32):
            j = order[start:start + 32]
            loss = F.cross_entropy(echo_model(Xe_tr[j]), ye_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    echo_model.eval()
    with torch.no_grad():
        p_abnormal = F.softmax(echo_model(Xe_te), dim=1)[:, 1].numpy()
    print("AUC for an EF below 50 percent:", round(roc_auc_score(test_pairs["abnormal"], p_abnormal), 3))

**Python note.** `if echo is None:` lets the cell run without a copy and print a message instead of an error. `echonet.frame_pairs` opens each video of the sample, takes the two traced frames and shrinks them to 64 × 64 pixels; reading 1400 videos takes a minute or two. The network and the loop are those of Section 5; only the number of outputs changes. The videos stay in your Colab session or on your computer.

## Application challenges by field

| Field | Challenge |
|---|---|
| Heart failure | Add the group of previous myocardial infarction with `acdc.pairs(("NOR", "DCM", "HCM", "MINF"))` and train `CmrCNN(n_classes=4)`. Which groups does the network confuse, and why could DCM and a large infarction look alike on the middle slices? |
| Interventional cardiology | Train the network with 15, 30 and 45 training patients and plot the accuracy on the 30 test patients against the number of training patients. How many examinations would a new laboratory need to collect before a network like this is worth training? |
| Electrophysiology | Read the number of frames per cardiac cycle, NbFrame, of each test patient with `acdc.read_info` and compare the errors of the network in examinations with fewer and with more than 20 frames. Why do the heart rate and the gating to the ECG matter for cine images? |
| Cardiovascular imaging | Use five middle slices per patient with `acdc.pairs(GROUPS, n_slices=5)` and train the network again. Does it now recognise patient 134? What would a network that reads the whole stack, from the base to the apex, need? |
| Preventive cardiology | Screening of the relatives of patients with HCM must not miss the disease. On the validation patients, choose the lowest probability of HCM that still finds every patient with HCM, apply it to the test patients and report the recall and the precision of HCM. |
| Cardio-oncology | Serial ejection fractions guide cardio-oncology. Plot the EF of each test patient from the contours against the probability of DCM given by the network. Would the network notice a fall of the EF from 60 to 50 percent as early as the measurement? |
| Intensive care | Time the prediction of the 90 test slices on your computer and compute the slices per second. How long would the network need for a whole stack of 10 slices with 30 frames each? |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 3, described on the [week page](README.md), compares the network of Section 5 with the rules of ACDC on the 30 test patients.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>